# Evaluate Vision Models on Size Illusions

In [ ]:
!pip3 install -q torch torchvision transformers pillow datasets matplotlib pandas

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image

PROJECT_ROOT = Path(os.environ.get('CS153_PROJECT_ROOT', '/cs/cs153/projects/size_illusion_project')).resolve()
DATA_ROOT = Path(os.environ.get('CS153_DATA_ROOT', '/cs/cs153/data/size_illusion_datasets')).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

CODE_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from data.generate_illusions import generate_dataset
from datasets.synthetic_loader import SyntheticIllusionDataset, ID_TO_LABEL
from datasets.illusionvqa_loader import load_illusionvqa_size, to_eval_examples
from models.cnn_model import build_resnet50, build_efficientnet_b0, extract_features, train_probe, predict_probe
from models.vit_model import build_vit_b16
from models.clip_eval import build_clip, clip_predict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT =', DATA_ROOT)
device


In [ ]:
data_root = DATA_ROOT / 'synthetic_illusions'
data_root.mkdir(parents=True, exist_ok=True)
generate_dataset(data_root, 'train', 2400, with_context=False, seed=10)
generate_dataset(data_root, 'test', 900, with_context=True, seed=11)
print('synthetic data ready at', data_root)


In [ ]:
resnet, res_tfm, res_dim = build_resnet50(device)
effnet, eff_tfm, eff_dim = build_efficientnet_b0(device)
vit, vit_tfm, vit_dim = build_vit_b16(device)
clip_model, clip_processor = build_clip(device)

In [ ]:
train_res = SyntheticIllusionDataset(data_root, "train", transform=res_tfm)
test_res = SyntheticIllusionDataset(data_root, "test", transform=res_tfm)
x_tr_r, y_tr, _ = extract_features(resnet, train_res, device)
x_te_r, y_te, rows_te = extract_features(resnet, test_res, device)
head_r = train_probe(x_tr_r, y_tr, res_dim, epochs=15, device=device)
pred_r, _ = predict_probe(head_r.to("cpu"), x_te_r)

train_eff = SyntheticIllusionDataset(data_root, "train", transform=eff_tfm)
test_eff = SyntheticIllusionDataset(data_root, "test", transform=eff_tfm)
x_tr_e, y_tr_e, _ = extract_features(effnet, train_eff, device)
x_te_e, y_te_e, _ = extract_features(effnet, test_eff, device)
head_e = train_probe(x_tr_e, y_tr_e, eff_dim, epochs=15, device=device)
pred_e, _ = predict_probe(head_e.to("cpu"), x_te_e)

train_vit = SyntheticIllusionDataset(data_root, "train", transform=vit_tfm)
test_vit = SyntheticIllusionDataset(data_root, "test", transform=vit_tfm)
x_tr_v, y_tr_v, _ = extract_features(vit, train_vit, device)
x_te_v, y_te_v, _ = extract_features(vit, test_vit, device)
head_v = train_probe(x_tr_v, y_tr_v, vit_dim, epochs=15, device=device)
pred_v, _ = predict_probe(head_v.to("cpu"), x_te_v)

In [ ]:
def acc_and_bias(pred_ids, gt_ids):
    n = len(gt_ids)
    acc = (pred_ids == gt_ids).float().mean().item()
    illusion_bias = (pred_ids != gt_ids).float().mean().item()
    return acc, illusion_bias

def label_from_text(x):
    t = str(x).lower()
    if "left" in t:
        return "left"
    if "right" in t:
        return "right"
    if "equal" in t or "same" in t:
        return "equal"
    return None

In [ ]:
train_rows = SyntheticIllusionDataset(data_root, "train", transform=res_tfm).rows
gt_train = torch.tensor([{"left": 0, "right": 1, "equal": 2}[r["ground_truth"]] for r in train_rows])
x_train_r, _, _ = extract_features(resnet, SyntheticIllusionDataset(data_root, "train", transform=res_tfm), device)
x_train_e, _, _ = extract_features(effnet, SyntheticIllusionDataset(data_root, "train", transform=eff_tfm), device)
x_train_v, _, _ = extract_features(vit, SyntheticIllusionDataset(data_root, "train", transform=vit_tfm), device)
pred_train_r, _ = predict_probe(head_r.to("cpu"), x_train_r)
pred_train_e, _ = predict_probe(head_e.to("cpu"), x_train_e)
pred_train_v, _ = predict_probe(head_v.to("cpu"), x_train_v)
no_ill_acc_r, _ = acc_and_bias(pred_train_r, gt_train)
no_ill_acc_e, _ = acc_and_bias(pred_train_e, gt_train)
no_ill_acc_v, _ = acc_and_bias(pred_train_v, gt_train)

test_gt = torch.tensor([{"left": 0, "right": 1, "equal": 2}[r["ground_truth"]] for r in rows_te])
ill_acc_r, ill_bias_r = acc_and_bias(pred_r, test_gt)
ill_acc_e, ill_bias_e = acc_and_bias(pred_e, test_gt)
ill_acc_v, ill_bias_v = acc_and_bias(pred_v, test_gt)

clip_train = SyntheticIllusionDataset(data_root, "train", transform=None)
clip_test = SyntheticIllusionDataset(data_root, "test", transform=None)
clip_pred_train, clip_gt_train = [], []
for i in range(len(clip_train)):
    im, y, _ = clip_train[i]
    p, _, _ = clip_predict(clip_model, clip_processor, im, device=device)
    clip_pred_train.append({"left": 0, "right": 1, "equal": 2}[p])
    clip_gt_train.append(y)
clip_pred_test, clip_gt_test = [], []
for i in range(len(clip_test)):
    im, y, _ = clip_test[i]
    p, _, _ = clip_predict(clip_model, clip_processor, im, device=device)
    clip_pred_test.append({"left": 0, "right": 1, "equal": 2}[p])
    clip_gt_test.append(y)
clip_pred_train = torch.tensor(clip_pred_train)
clip_gt_train = torch.tensor(clip_gt_train)
clip_pred_test = torch.tensor(clip_pred_test)
clip_gt_test = torch.tensor(clip_gt_test)
no_ill_acc_c, _ = acc_and_bias(clip_pred_train, clip_gt_train)
ill_acc_c, ill_bias_c = acc_and_bias(clip_pred_test, clip_gt_test)

synthetic_results = pd.DataFrame(
    [
        ["ResNet50", no_ill_acc_r, ill_acc_r, ill_bias_r],
        ["EfficientNet-B0", no_ill_acc_e, ill_acc_e, ill_bias_e],
        ["ViT-B/16", no_ill_acc_v, ill_acc_v, ill_bias_v],
        ["CLIP", no_ill_acc_c, ill_acc_c, ill_bias_c],
    ],
    columns=["Model", "No-illusion acc", "Illusion acc", "Illusion bias"],
)
synthetic_results

In [ ]:
ivqa = load_illusionvqa_size(split="test")
ivqa_examples = to_eval_examples(ivqa)
ivqa_examples = [x for x in ivqa_examples if label_from_text(x.get("answer")) is not None]
print("IllusionVQA size examples:", len(ivqa_examples))

In [ ]:
def eval_ivqa_model(name):
    preds = []
    gts = []
    for ex in ivqa_examples:
        image = ex["image"].convert("RGB") if hasattr(ex["image"], "convert") else Image.fromarray(ex["image"])
        gt = label_from_text(ex["answer"])
        if gt is None:
            continue
        gts.append({"left": 0, "right": 1, "equal": 2}[gt])
        if name == "CLIP":
            p, _, _ = clip_predict(clip_model, clip_processor, image, device=device)
            preds.append({"left": 0, "right": 1, "equal": 2}[p])
        elif name == "ResNet50":
            x = res_tfm(image).unsqueeze(0).to(device)
            with torch.no_grad():
                f = resnet(x).cpu()
            p, _ = predict_probe(head_r.to("cpu"), f)
            preds.append(int(p[0]))
        elif name == "EfficientNet-B0":
            x = eff_tfm(image).unsqueeze(0).to(device)
            with torch.no_grad():
                f = effnet(x).cpu()
            p, _ = predict_probe(head_e.to("cpu"), f)
            preds.append(int(p[0]))
        else:
            x = vit_tfm(image).unsqueeze(0).to(device)
            with torch.no_grad():
                f = vit(x).cpu()
            p, _ = predict_probe(head_v.to("cpu"), f)
            preds.append(int(p[0]))
    preds = torch.tensor(preds)
    gts = torch.tensor(gts)
    acc, bias = acc_and_bias(preds, gts)
    return acc, bias

ivqa_rows = []
for name in ["ResNet50", "EfficientNet-B0", "ViT-B/16", "CLIP"]:
    acc, bias = eval_ivqa_model(name)
    ivqa_rows.append([name, None, acc, bias])
ivqa_results = pd.DataFrame(ivqa_rows, columns=["Model", "No-illusion acc", "Illusion acc", "Illusion bias"])
ivqa_results

In [ ]:
rows = SyntheticIllusionDataset(data_root, "test", transform=None).rows
pred_lbl_res = [ID_TO_LABEL[int(x)] for x in pred_r]
gt_lbl = [r["ground_truth"] for r in rows]
correct_idx = [i for i, (p, g) in enumerate(zip(pred_lbl_res, gt_lbl)) if p == g][:3]
fooled_idx = [i for i, (p, g) in enumerate(zip(pred_lbl_res, gt_lbl)) if p != g][:3]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for col, idx in enumerate(correct_idx):
    row = rows[idx]
    im = Image.open(data_root / row["image_path"])
    axes[0, col].imshow(im)
    axes[0, col].set_title(f"correct\nGT={row['ground_truth']} Pred={pred_lbl_res[idx]}")
    axes[0, col].axis("off")
for col, idx in enumerate(fooled_idx):
    row = rows[idx]
    im = Image.open(data_root / row["image_path"])
    axes[1, col].imshow(im)
    axes[1, col].set_title(f"fooled\nGT={row['ground_truth']} Pred={pred_lbl_res[idx]}")
    axes[1, col].axis("off")
plt.tight_layout()
plt.show()